In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

In [ ]:
daily_metrics = pd.read_csv('data/by_day_all_days.csv')
daily_metrics.head()

In [ ]:
videos = pd.read_csv('data/by_video.csv')
videos.head()

In [ ]:
video_metadata = pd.read_csv('data/video_metadata.csv')
video_metadata.head()

In [ ]:
videos_metrics_metadata = videos.merge(video_metadata, on='video')
videos_metrics_metadata.head()

In [ ]:
videos_metrics_metadata.drop(columns=['viewCount', 'likeCount', 'commentCount'], inplace=True)
videos_metrics_metadata.head()

In [ ]:
daily_metrics_copy = daily_metrics.copy()
videos_metrics_metadata_copy = videos_metrics_metadata.copy()

In [ ]:
daily_metrics_copy.head()

In [ ]:
daily_metrics_copy['day'] = pd.to_datetime(daily_metrics_copy['day'])

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(daily_metrics_copy['day'], daily_metrics_copy['engagedViews'])

plt.xlabel("Day")
plt.ylabel("Views")
plt.title("Views by Day")

plt.gca().tick_params(axis='x', rotation=30, labelsize=8)

plt.show()

In [ ]:
avg = daily_metrics_copy['engagedViews'].mean()
avg

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(daily_metrics_copy['day'], daily_metrics_copy['engagedViews'])

plt.axhline(avg, color='gray', linestyle='--', linewidth=1)

plt.xlabel("Day")
plt.ylabel("Views")
plt.title("Views by Day")

plt.gca().tick_params(axis='x', rotation=30, labelsize=8)

plt.show()

In [ ]:
daily_metrics_copy = daily_metrics_copy.set_index('day')
monthly_avg = daily_metrics_copy['engagedViews'].resample('MS').mean()
monthly_avg
daily_metrics_copy.reset_index(inplace=True)

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(daily_metrics_copy['day'], daily_metrics_copy['engagedViews'])

plt.axhline(avg, color='gray', linestyle='--', linewidth=1)

plt.plot(monthly_avg.index, monthly_avg.values, label='Monthly View Average', color='black', marker='o')


plt.xlabel("Day")
plt.ylabel("Views")
plt.title("Views by Day")

plt.gca().tick_params(axis='x', rotation=30, labelsize=8)

plt.show()

In [ ]:
import matplotlib.dates as mdates

daily_metrics_copy['netSubscribers'] = daily_metrics_copy['subscribersGained'] - daily_metrics_copy['subscribersLost']

plt.figure(figsize=(15, 5))
plt.plot(daily_metrics_copy['day'], daily_metrics_copy['netSubscribers'])

plt.xlabel("Day")      # x-axis title
plt.ylabel("Net Subscribers")      # y-axis title
plt.title("Net Subscribers by Day")  # chart title

# Tick marks at the start of each month
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.gca().tick_params(axis='x', rotation=30, labelsize=8)

# Average line
avg = daily_metrics_copy['netSubscribers'].mean()
plt.axhline(avg, color='gray', linestyle='--', linewidth=1, label=f'Average: {avg:.1f}')

#Add monthly average as well
daily_metrics_copy = daily_metrics_copy.set_index('day')
monthly_avg = daily_metrics_copy['netSubscribers'].resample('MS').mean()
daily_metrics_copy.reset_index(inplace=True)

plt.plot(monthly_avg.index, monthly_avg.values, label='Monthly Average', color='black', linewidth=1, marker='o')

plt.show()

In [ ]:
# Let's also look to see how average watch time has been changing over time
plt.figure(figsize=(15, 5))
plt.plot(daily_metrics_copy['day'], daily_metrics_copy['averageViewPercentage'])

plt.xlabel("Day")      # x-axis title
plt.ylabel("averageViewPercentage")      # y-axis title
plt.title("Average View Percentage by Day")  # chart title

# Tick marks at the start of each month
plt.gca().xaxis.set_major_locator(mdates.MonthLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

plt.gca().tick_params(axis='x', rotation=30, labelsize=8)

plt.show()

Average number of views is about 150, but in recent months, the daily average has been higher. The channel is getting on average about 5 subscribers a day, and around 20% average watch time. There was a large spike in April - July of 2025.

In [ ]:
videos_metrics_metadata_copy.head()

In [ ]:
from transformers import pipeline

MODEL = 'facebook/bart-large-mnli'

TOPICS = [
'Python Code Tutorials',
'SQL Tutorials',
'AI',
'Machine Learning',
'Data Analysis Frameworks',
'Career Advice/Experience',
'Statistics Tutorials'
]

CONFIDENCE_THRESHOLD = 0.35
FALLBACK_LABEL = 'Uncategorized'

classifier = pipeline('zero-shot-classification', model=MODEL, device=-1)

In [ ]:
def classify_video(title, description):
    text   = str(title) + '. ' + str(description or '')[:400]
    result = classifier(text, candidate_labels=TOPICS, multi_label=False)
    label  = result['labels'][0]
    score  = result['scores'][0]
    return label if score >= CONFIDENCE_THRESHOLD else FALLBACK_LABEL

In [ ]:
videos_metrics_metadata_copy['video_topic'] = videos_metrics_metadata_copy.apply(lambda x: classify_video(x['title'], x['description']), axis=1)

In [ ]:
videos_metrics_metadata_copy[['title', 'description', 'video_topic']]

In [ ]:
videos_metrics_metadata_copy[videos_metrics_metadata_copy['video_topic'] == 'Uncategorized'][['title', 'description', 'video_topic']]

In [ ]:
videos_metrics_metadata_copy.loc[3, 'video_topic'] = 'Data Analysis Frameworks'
videos_metrics_metadata_copy.loc[10, 'video_topic'] = 'Python Code Tutorials'
videos_metrics_metadata_copy.loc[20, 'video_topic'] = 'Data Analysis Frameworks'
videos_metrics_metadata_copy.loc[24, 'video_topic'] = 'Python Code Tutorials'
videos_metrics_metadata_copy.loc[35, 'video_topic'] = 'Data Analysis Frameworks'
videos_metrics_metadata_copy.loc[37, 'video_topic'] = 'Data Analysis Frameworks'

In [ ]:
videos_metrics_metadata_copy_grouped = videos_metrics_metadata_copy['video_topic'].value_counts()
videos_metrics_metadata_copy_grouped

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(videos_metrics_metadata_copy_grouped.index, videos_metrics_metadata_copy_grouped.values, color='#3498db')
plt.xlabel("Video Topic")
plt.ylabel("Count")
plt.title("Number of Videos per Topic")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
videos_metrics_metadata_copy['publishedAt'] = pd.to_datetime(videos_metrics_metadata_copy['publishedAt'])
videos_metrics_metadata_copy['month'] = videos_metrics_metadata_copy['publishedAt'].dt.to_period('M').dt.to_timestamp()

monthly_topic_counts = videos_metrics_metadata_copy.groupby(['month', 'video_topic']).size().reset_index(name='count')

monthly_topic_counts

In [ ]:
pivot_df = monthly_topic_counts.pivot(index='month', columns='video_topic', values='count').fillna(0)
pivot_df

In [ ]:
plt.figure(figsize=(15, 6))
pivot_df.plot(kind='bar', stacked=True, figsize=(15, 6))

plt.xlabel("Month")
plt.ylabel("Number of Videos")
plt.title("Video Topic Mix Over Time")
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

We can see at the beginning of the channel and when it was experiencing the most views, a lot of Python tutorials were being created. But recently, it looks like the channel has shifted to more to frameworks, career advice, and even a couple sql tutorials thrown in there.

In [ ]:
videos_metrics_metadata_copy['netSubscribers'] = videos_metrics_metadata_copy['subscribersGained'] - videos_metrics_metadata_copy['subscribersLost']

subs_by_topic = videos_metrics_metadata_copy.groupby('video_topic')['netSubscribers'].sum().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(subs_by_topic.index, subs_by_topic.values, color="#C2185B")
plt.title("Net Subscribers by Topic", fontsize=14, fontweight="bold")
plt.xlabel("Net Subscribers Gained", fontsize=11)
plt.ylabel("Topic", fontsize=11)
plt.tight_layout()

plt.show()

In [ ]:
videos_metrics_metadata_copy_sub_conversion = videos_metrics_metadata_copy.groupby('video_topic')[['netSubscribers', 'engagedViews']].sum().reset_index()
videos_metrics_metadata_copy_sub_conversion

videos_metrics_metadata_copy_sub_conversion['sub_conv_rate'] = videos_metrics_metadata_copy_sub_conversion['netSubscribers'] / videos_metrics_metadata_copy_sub_conversion['engagedViews'] * 1000

videos_metrics_metadata_copy_sub_conversion

In [ ]:
videos_metrics_metadata_copy_sub_conversion.sort_values(by='sub_conv_rate', ascending=True, inplace=True)

plt.figure(figsize=(10, 6))
plt.barh(videos_metrics_metadata_copy_sub_conversion['video_topic'], videos_metrics_metadata_copy_sub_conversion['sub_conv_rate'], color="#C2185B")
plt.title("Net Subscribers Conv Rate Per 1000 Views by Topic", fontsize=14, fontweight="bold")
plt.xlabel("Net Subscribers Conv Rate", fontsize=11)
plt.ylabel("Topic", fontsize=11)
plt.tight_layout()

plt.show()

In [ ]:
career_videos = videos_metrics_metadata_copy[videos_metrics_metadata_copy['video_topic'] == 'Career Advice/Experience'].copy()
career_videos['sub_conv_rate'] = career_videos['netSubscribers'] / career_videos['engagedViews'] * 1000
career_videos[['title', 'netSubscribers', 'engagedViews', 'sub_conv_rate']].sort_values('sub_conv_rate', ascending=False)

Python code tutorial videos had the highest number of subscribers gained. However, that topic is the most common one among the videos on the channel. It would have more videos, and in turn more views, so more subscribers. When looking at conversion rates however, career advice jumps to number 1. Within that topic though, some videos outperform others.

In [ ]:
videos_metrics_metadata_copy.head()

In [ ]:
videos_metrics_metadata_copy['watch_hrs'] = videos_metrics_metadata_copy['estimatedMinutesWatched'] / 60

topic_wt = videos_metrics_metadata_copy.groupby('video_topic')['watch_hrs'].sum().sort_values(ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(topic_wt.index, topic_wt.values, color="#C2185B")
plt.title("Estimated Watch Hours by Topic", fontsize=14, fontweight="bold")
plt.xlabel("Estimated Watch Hours", fontsize=11)
plt.ylabel("Topic", fontsize=11)
plt.tight_layout()

plt.show()

In [ ]:
videos_metrics_metadata_copy.boxplot(column = 'averageViewPercentage', by='video_topic', vert=False, figsize=(10,6))

plt.xlabel("Average View Duration")
plt.ylabel("Video Topic")
plt.title("Average View Duration by Topic")
plt.suptitle('')  # removes the automatic subtitle pandas adds
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
total_engagement = videos_metrics_metadata_copy.groupby('video_topic')[['likes', 'comments', 'shares', 'engagedViews']].sum().reset_index()

total_engagement['total_engagement'] = total_engagement['likes'] + total_engagement['comments'] + total_engagement['shares']
total_engagement['engagement_rate'] = ((total_engagement['likes'] + total_engagement['comments'] + total_engagement['shares']) / total_engagement['engagedViews']) * 100

In [ ]:
top15 = total_engagement.sort_values('total_engagement', ascending=True).tail(15)

plt.figure(figsize=(10, 6))
plt.barh(top15['video_topic'], top15['total_engagement'], color="#C2185B")
plt.title("Total Engagement By Video (Top 15)", fontsize=14, fontweight="bold")
plt.xlabel("Total Engagement", fontsize=11)
plt.ylabel("Video", fontsize=11)
plt.tight_layout()

plt.show()

In [ ]:
top15 = total_engagement.sort_values('engagement_rate', ascending=True).tail(15)

plt.figure(figsize=(10, 6))
plt.barh(top15['video_topic'], top15['engagement_rate'], color="#C2185B")
plt.title("Engagement Rate by Video (Top 15)", fontsize=14, fontweight="bold")
plt.xlabel("Engagement Rate", fontsize=11)
plt.ylabel("Video", fontsize=11)
plt.tight_layout()

plt.show()

In [ ]:
videos_metrics_metadata_copy_career_only = videos_metrics_metadata_copy[videos_metrics_metadata_copy['video_topic'] == 'Career Advice/Experience']

videos_metrics_metadata_copy_career_only['total_engagement'] = videos_metrics_metadata_copy_career_only['likes'] + videos_metrics_metadata_copy_career_only['comments'] + videos_metrics_metadata_copy_career_only['shares']
videos_metrics_metadata_copy_career_only['engagement_rate'] = ((videos_metrics_metadata_copy_career_only['likes'] + videos_metrics_metadata_copy_career_only['comments'] + videos_metrics_metadata_copy_career_only['shares']) / videos_metrics_metadata_copy_career_only['engagedViews']) * 100

videos_metrics_metadata_copy_career_only

In [ ]:
threshold = videos_metrics_metadata_copy_career_only['engagement_rate'].quantile(0.75)
videos_metrics_metadata_copy_career_only['high_engagement'] = videos_metrics_metadata_copy_career_only['engagement_rate'] >= threshold
threshold

In [ ]:
videos_metrics_metadata_copy_career_only[['engagement_rate', 'high_engagement']]

In [ ]:
videos_metrics_metadata_copy_career_only['day_of_week'] = videos_metrics_metadata_copy_career_only['publishedAt'].dt.day_name()

dow_comparison = pd.crosstab(videos_metrics_metadata_copy_career_only['day_of_week'],
videos_metrics_metadata_copy_career_only['high_engagement'],
normalize='columns') * 100

dow_comparison

In [ ]:
traffic_source_df = pd.read_csv('data/by_traffic_source.csv')
traffic_source_df

In [ ]:
traffic_source_df = traffic_source_df.sort_values(by='views', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(traffic_source_df['insightTrafficSourceType'], traffic_source_df['views'], color="#C2185B")
plt.title("Views By Traffic Source", fontsize=14, fontweight="bold")
plt.xlabel("Views", fontsize=11)
plt.ylabel("Traffic Source", fontsize=11)
plt.tight_layout()

plt.show()

In [ ]:
traffic_source_df_sorted = traffic_source_df.sort_values('averageViewDuration', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(traffic_source_df_sorted['insightTrafficSourceType'], traffic_source_df_sorted['averageViewDuration'], color="#C2185B")
plt.title("Average View Duration By Traffic Source", fontsize=14, fontweight="bold")
plt.xlabel("Average View Duration", fontsize=11)
plt.ylabel("Traffic Source", fontsize=11)
plt.tight_layout()

plt.show()